In [2]:
import subprocess
from pathlib import Path

structures_dir = Path("/content/structures")
structures_dir.mkdir(exist_ok=True)

subprocess.run(["apt-get", "install", "-q", "-y", "dssp"], check=True)
print("Ready.")

Ready.


In [3]:
import shutil
from pathlib import Path

structures_dir = Path("/content/structures")
structures_dir.mkdir(exist_ok=True)

# TNXB, FREM2, FRAS1 — externally predicted, uploaded as full-length single-chain PDBs
for gene in ["TNXB", "FREM2", "FRAS1"]:
    src = Path(f"/content/{gene}.pdb")  # upload these to /content first via the Files pane
    dst = structures_dir / f"{gene}.pdb"
    if src.exists():
        shutil.copy(src, dst)
        print(f"{gene}: copied ({dst.stat().st_size/1024:.1f} KB)")
    else:
        print(f"{gene}: upload {src.name} to /content first")

print("\nFiles in structures/:")
for f in sorted(structures_dir.glob("*.pdb")):
    print(" ", f.name)

TNXB: copied (2546.1 KB)
FREM2: copied (1955.6 KB)
FRAS1: copied (2455.8 KB)

Files in structures/:
  FRAS1.pdb
  FREM2.pdb
  TNXB.pdb


In [4]:
import pandas as pd

rows = [
    {"gene": "TNXB",  "panel": "CAKUT", "accession": "P22105", "length": 4243, "entry_name": "Tenascin-X"},
    {"gene": "FREM2", "panel": "CAKUT", "accession": "O94898", "length": 2998, "entry_name": "FRAS1-related extracellular matrix protein 2"},
    {"gene": "FRAS1", "panel": "CAKUT", "accession": "Q86WK7", "length": 4007, "entry_name": "Extracellular matrix protein FRAS1"},
]
acc_df = pd.DataFrame(rows)
acc_df.to_csv("/content/uniprot_accessions.csv", index=False)
print(acc_df)

    gene  panel accession  length  \
0   TNXB  CAKUT    P22105    4243   
1  FREM2  CAKUT    O94898    2998   
2  FRAS1  CAKUT    Q86WK7    4007   

                                     entry_name  
0                                    Tenascin-X  
1  FRAS1-related extracellular matrix protein 2  
2            Extracellular matrix protein FRAS1  


In [5]:
import requests, time, pandas as pd

acc_df = pd.read_csv("/content/uniprot_accessions.csv")

keep_types = {
    "Domain", "Region", "Active site", "Binding site", "Site",
    "Signal", "Propeptide", "Transit peptide", "Chain",
    "Transmembrane", "Coiled coil", "Compositional bias",
    "Modified residue", "Disulfide bond", "Motif",
}

rows = []
for i, row in enumerate(acc_df.itertuples(), 1):
    gene, acc = row.gene, row.accession
    try:
        r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json", timeout=20)
        r.raise_for_status()
        data     = r.json()
        features = data.get("features", [])
        length   = data.get("sequence", {}).get("length")
        n = 0
        for f in features:
            ftype = f.get("type", "")
            if ftype not in keep_types:
                continue
            loc   = f.get("location", {})
            start = loc.get("start", {}).get("value")
            end   = loc.get("end",   {}).get("value")
            rows.append({
                "gene": gene, "panel": row.panel, "accession": acc, "length": length,
                "feature_type": ftype, "description": f.get("description", ""),
                "start": start, "end": end,
            })
            n += 1
        print(f"[{i}] {gene:<12} {acc}  {length:>5} aa  {n} features")
    except Exception as e:
        print(f"[{i}] {gene:<12} {acc}  ERROR: {e}")
    time.sleep(0.15)

feat_df = pd.DataFrame(rows)
feat_df.to_csv("/content/uniprot_features.csv", index=False)
print(f"\nTotal feature rows: {len(feat_df)}")
print(feat_df["feature_type"].value_counts().to_string())

[1] TNXB         P22105   4244 aa  128 features
[2] FREM2        O94898   1065 aa  16 features
[3] FRAS1        Q86WK7    504 aa  13 features

Total feature rows: 157
feature_type
Disulfide bond        64
Domain                60
Region                15
Compositional bias     8
Signal                 3
Chain                  3
Transmembrane          2
Motif                  1
Modified residue       1


In [6]:
import subprocess, pandas as pd
from pathlib import Path

structures_dir = Path("/content/structures")
rsa_dir = Path("/content/rsa")
rsa_dir.mkdir(exist_ok=True)

max_asa = {
    "ALA":129.0,"ARG":274.0,"ASN":195.0,"ASP":193.0,"CYS":167.0,
    "GLN":225.0,"GLU":223.0,"GLY":104.0,"HIS":224.0,"ILE":197.0,
    "LEU":201.0,"LYS":236.0,"MET":224.0,"PHE":240.0,"PRO":159.0,
    "SER":155.0,"THR":172.0,"TRP":285.0,"TYR":263.0,"VAL":174.0,
}
ss_map = {"H":"helix","G":"helix","I":"helix","E":"sheet","B":"sheet",
          "T":"loop","S":"loop"," ":"loop","-":"loop"}
aa3_map = {
    "A":"ALA","R":"ARG","N":"ASN","D":"ASP","C":"CYS","Q":"GLN",
    "E":"GLU","G":"GLY","H":"HIS","I":"ILE","L":"LEU","K":"LYS",
    "M":"MET","F":"PHE","P":"PRO","S":"SER","T":"THR","W":"TRP",
    "Y":"TYR","V":"VAL",
}

# Dummy CRYST1 record — only used for files with no header at all
# (bare ColabFold/AF3 output starting directly with ATOM, e.g. CUBN/FAT1).
# Files that already start with HEADER (e.g. SLC9A3R1 from AFDB) are left untouched.
DUMMY_CRYST1 = "CRYST1    1.000    1.000    1.000  90.00  90.00  90.00 P 1           1\n"

def parse_dssp(text):
    rows, in_data = [], False
    for line in text.splitlines():
        if "#  RESIDUE AA STRUCTURE" in line:
            in_data = True; continue
        if not in_data or len(line) < 38:
            continue
        if line[13] == "!":
            continue
        try:
            resnum = int(line[5:10].strip())
            aa     = line[13].strip()
            ss_raw = line[16]
            asa    = float(line[35:38].strip())
        except (ValueError, IndexError):
            continue
        rows.append({"resnum": resnum, "aa": aa, "ss_raw": ss_raw,
                     "ss": ss_map.get(ss_raw, "loop"), "asa": asa})
    return rows

def extract_plddt(pdb_path):
    plddt = {}
    with open(pdb_path) as f:
        for line in f:
            if line[:4] == "ATOM" and line[12:16].strip() == "CA":
                try:
                    resnum = int(line[22:26].strip())
                    plddt[resnum] = float(line[60:66].strip())
                except ValueError:
                    continue
    return plddt

log = []
for pdb_path in sorted(structures_dir.glob("*.pdb")):
    gene = pdb_path.stem

    text = pdb_path.read_text()
    first_line = text.splitlines()[0] if text else ""
    if not first_line.startswith(("HEADER", "CRYST1")):
        pdb_path.write_text(DUMMY_CRYST1 + text)

    try:
        result = subprocess.run(
            ["mkdssp", "--output-format", "dssp", str(pdb_path)],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr.strip()[:200])

        dssp_rows = parse_dssp(result.stdout)
        if not dssp_rows:
            raise RuntimeError("no residues parsed")

        plddt_map = extract_plddt(pdb_path)

        rows = []
        for r in dssp_rows:
            rn      = r["resnum"]
            resname = aa3_map.get(r["aa"])
            max_a   = max_asa.get(resname) if resname else None
            rsa     = round(r["asa"] / max_a, 4) if (r["asa"] is not None and max_a) else None
            plddt   = plddt_map.get(rn)
            rows.append({
                "gene": gene, "fragment": "1", "resnum": rn, "aa": r["aa"],
                "ss_raw": r["ss_raw"], "ss": r["ss"], "asa": r["asa"],
                "rsa": rsa, "plddt": round(plddt, 2) if plddt is not None else None,
            })

        df = pd.DataFrame(rows)
        df.to_csv(rsa_dir / f"{gene}.csv", index=False)
        print(f"{gene:<12} {len(rows):>5} residues")
        log.append({"gene": gene, "residues": len(rows), "status": "ok"})

    except Exception as e:
        print(f"{gene:<12} ERROR: {e}")
        log.append({"gene": gene, "residues": 0, "status": f"error: {e}"})

pd.DataFrame(log).to_csv("/content/dssp_log.csv", index=False)
print(f"\n{pd.DataFrame(log)['status'].value_counts().to_string()}")

FRAS1         4008 residues
FREM2         3169 residues
TNXB          4242 residues

status
ok    3


In [7]:
import pandas as pd
from pathlib import Path

rsa_dir  = Path("/content/rsa")
full_dir = Path("/content/rsa_full")
full_dir.mkdir(exist_ok=True)

for f in sorted(rsa_dir.glob("*.csv")):
    df = pd.read_csv(f)
    df.to_csv(full_dir / f.name, index=False)
    print(f"{f.stem:<12} {len(df)} residues")

FRAS1        4008 residues
FREM2        3169 residues
TNXB         4242 residues


In [8]:
template = """<!DOCTYPE html>
<html lang="en"><head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{gene} — NephVar Biophysical</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;500;600;700&family=DM+Mono:wght@300;400&family=Instrument+Serif:ital@0;1&display=swap" rel="stylesheet">
<style>
:root{{--serif:'Instrument Serif',Georgia,serif;--sans:'Syne',system-ui,sans-serif;
--mono:'DM Mono',ui-monospace,monospace;--accent:#1f6fa8;--ink:#16202b;
--soft:#5d6b78;--faint:#8a96a3;--line:#e6e9ee;--bg:#fbfaf7;
--helix:#e07b18;--sheet:#1f6fa8;--loop:#c8ced4;
--buried:#c0392b;--exposed:#2ecc71}}
*{{box-sizing:border-box}}
body{{margin:0;font-family:var(--sans);color:var(--ink);background:var(--bg);-webkit-font-smoothing:antialiased}}
.wrap{{max-width:1240px;margin:0 auto;padding:40px 22px 70px}}
.crumb{{font-family:var(--mono);font-size:11px;letter-spacing:.14em;text-transform:uppercase;color:var(--faint);margin-bottom:16px}}
.crumb a{{color:var(--accent);text-decoration:none}}.crumb a:hover{{text-decoration:underline}}
h1{{font-family:var(--serif);font-weight:400;font-size:40px;margin:0 0 4px}}
h1 em{{font-style:italic;color:var(--accent)}}
.sub{{color:var(--soft);font-size:14px;margin-bottom:28px;line-height:1.6}}
.map-wrap{{background:#fff;border:1px solid var(--line);border-radius:6px;padding:24px;margin-bottom:32px;overflow-x:auto}}
.map-title{{font-family:var(--mono);font-size:10.5px;text-transform:uppercase;letter-spacing:.1em;color:var(--soft);margin-bottom:16px}}
svg.protmap{{display:block;width:100%;min-width:600px;height:180px}}
.legend{{display:flex;flex-wrap:wrap;gap:16px;margin-top:14px;font-family:var(--mono);font-size:10.5px;color:var(--soft)}}
.legend span{{display:flex;align-items:center;gap:6px}}
.swatch{{width:12px;height:12px;border-radius:2px;display:inline-block;flex-shrink:0}}
.controls{{display:flex;flex-wrap:wrap;gap:10px;align-items:center;margin-bottom:14px}}
input[type=text],select{{font-family:var(--sans);font-size:13px;padding:9px 12px;border:1px solid var(--line);border-radius:4px;background:#fff;color:var(--ink)}}
input[type=text]{{flex:1;min-width:200px}}
.count{{font-family:var(--mono);font-size:11px;color:var(--faint);margin-left:auto}}
table{{width:100%;border-collapse:collapse;font-size:13px;background:#fff;border:1px solid var(--line);border-radius:4px;overflow:hidden}}
th{{font-family:var(--mono);text-align:left;padding:11px 12px;background:#f6f5f1;font-weight:400;font-size:10.5px;letter-spacing:.1em;text-transform:uppercase;color:var(--soft);cursor:pointer;white-space:nowrap}}
th:hover{{color:var(--ink)}}
td{{padding:9px 12px;border-top:1px solid var(--line)}}
tbody tr:hover td{{background:#f7f9fb}}
.mono{{font-family:var(--mono);font-size:11.5px}}
.pill{{font-family:var(--mono);display:inline-block;padding:2px 8px;border-radius:99px;font-size:10px;color:#fff}}
.pill.helix{{background:var(--helix)}}.pill.sheet{{background:var(--sheet)}}.pill.loop{{background:#8a96a3}}
.pill.buried{{background:var(--buried)}}.pill.exposed{{background:var(--exposed)}}
.empty{{padding:40px;text-align:center;color:var(--faint)}}
#tip{{position:fixed;display:none;background:#16202b;color:#fff;font-family:var(--mono);
font-size:11px;padding:8px 12px;border-radius:4px;pointer-events:none;z-index:20;line-height:1.6}}
</style></head>
<body><div class="wrap">
<div class="crumb"><a href="../">NephVar</a> / <a href="./">Biophysical</a> / {gene}</div>
<h1>{gene} <em>{protein_name}</em></h1>
<div class="sub">{panel} panel &middot; {length} aa &middot;
UniProt <a href="https://www.uniprot.org/uniprot/{accession}" target="_blank"
style="color:var(--accent)">{accession}</a></div>

<div class="map-wrap">
  <div class="map-title">Protein map — domains / secondary structure / RSA / pLDDT</div>
  <svg class="protmap" id="protmap" viewBox="0 0 1000 180"></svg>
  <div class="legend">
    <span><span class="swatch" style="background:var(--helix)"></span>Helix</span>
    <span><span class="swatch" style="background:var(--sheet)"></span>Sheet</span>
    <span><span class="swatch" style="background:var(--loop)"></span>Loop / coil</span>
    <span><span class="swatch" style="background:#a0c4e8"></span>pLDDT confidence</span>
    <span><span class="swatch" style="background:var(--buried)"></span>Buried (RSA &lt; 0.25)</span>
    <span><span class="swatch" style="background:var(--exposed)"></span>Exposed (RSA &ge; 0.25)</span>
  </div>
</div>

<div class="controls">
  <input type="text" id="q" placeholder="Search position or amino acid…">
  <select id="ss"><option value="">All secondary structures</option>
    <option value="helix">Helix</option>
    <option value="sheet">Sheet</option>
    <option value="loop">Loop / coil</option>
  </select>
  <select id="dom"><option value="">All domains</option>{domain_options}</select>
  <span class="count" id="count"></span>
</div>
<table><thead><tr>
  <th data-k="resnum">Pos</th>
  <th data-k="aa">AA</th>
  <th data-k="rsa">RSA</th>
  <th data-k="ss">2° structure</th>
  <th data-k="plddt">pLDDT</th>
  <th data-k="buried">Burial</th>
  <th data-k="domain">Domain</th>
</tr></thead><tbody id="rows"></tbody></table>
<div class="empty" id="empty" style="display:none">No residues match.</div>
</div>
<div id="tip"></div>

<script>
const residues = {residue_json};
const domains  = {domain_json};
const seqlen   = {length};

const svg  = document.getElementById("protmap");
const ns   = "http://www.w3.org/2000/svg";
const W    = 1000, pad = 20, inner = W - pad * 2;
const xOf  = r => pad + ((r - 1) / Math.max(seqlen - 1, 1)) * inner;

const yDom = 12, domH = 14;
const ySS  = 36, ssH  = 12;
const yRSA = 62, rsaH = 32;
const yPLD = 108, pldH = 32;

function rect(attrs) {{
  const e = document.createElementNS(ns, "rect");
  for (const [k,v] of Object.entries(attrs)) e.setAttribute(k,v);
  return e;
}}
function text(attrs, label) {{
  const e = document.createElementNS(ns, "text");
  for (const [k,v] of Object.entries(attrs)) e.setAttribute(k,v);
  e.textContent = label;
  return e;
}}

[["Domains", yDom], ["2° Str.", ySS], ["RSA", yRSA], ["pLDDT", yPLD]].forEach(([label, y]) =>
  svg.appendChild(text({{x:2, y:y-1, fill:"#8a96a3", "font-size":"7",
    "font-family":"DM Mono,monospace"}}, label))
);

const domColors = ["#7ec8e3","#f4a261","#a8dadc","#c77dff","#90be6d",
                   "#f9c74f","#f94144","#43aa8b","#577590","#e9c46a"];
const domIdx = {{}};
domains.forEach(d => {{
  if (!d.start || !d.end) return;
  if (!(d.description in domIdx)) domIdx[d.description] = domColors[Object.keys(domIdx).length % domColors.length];
  const x = xOf(d.start), w = Math.max(2, xOf(d.end) - xOf(d.start));
  const r = rect({{x, y:yDom, width:w, height:domH, fill:domIdx[d.description], opacity:"0.85", rx:"2"}});
  const t = document.createElementNS(ns, "title");
  t.textContent = `${{d.description}} (${{d.start}}–${{d.end}})`;
  r.appendChild(t);
  svg.appendChild(r);
}});

residues.forEach(r => {{
  const x  = xOf(r.resnum);
  const ss = r.ss || "loop";
  const ssColor = ss === "helix" ? "#e07b18" : ss === "sheet" ? "#1f6fa8" : "#c8ced4";
  svg.appendChild(rect({{x, y:ySS, width:"1.5", height:ssH, fill:ssColor}}));

  if (r.rsa != null) {{
    const h = r.rsa * rsaH;
    svg.appendChild(rect({{x, y:yRSA+rsaH-h, width:"1.5", height:h,
      fill: r.rsa < 0.25 ? "#c0392b" : "#2ecc71", opacity:"0.7"}}));
  }}
  if (r.plddt != null) {{
    const h = (r.plddt / 100) * pldH;
    svg.appendChild(rect({{x, y:yPLD+pldH-h, width:"1.5", height:h, fill:"#a0c4e8", opacity:"0.8"}}));
  }}
}});

function domainOf(resnum) {{
  const hit = domains.find(d => d.type === "Domain" && resnum >= d.start && resnum <= d.end);
  return hit ? hit.description : "";
}}

const rows = residues.map(r => ({{
  ...r,
  domain: domainOf(r.resnum),
  buried: r.rsa == null ? "" : r.rsa < 0.25 ? "buried" : "exposed"
}}));

let sortKey = "resnum", sortAsc = true;
const tbody   = document.getElementById("rows");
const countEl = document.getElementById("count");
const emptyEl = document.getElementById("empty");

function render() {{
  const q   = document.getElementById("q").value.toLowerCase();
  const ss  = document.getElementById("ss").value;
  const dom = document.getElementById("dom").value;

  let data = rows.filter(r =>
    (!q   || String(r.resnum).includes(q) || (r.aa||"").toLowerCase().includes(q)) &&
    (!ss  || r.ss === ss) &&
    (!dom || r.domain === dom)
  );

  data.sort((a, b) => {{
    const av = a[sortKey] ?? "", bv = b[sortKey] ?? "";
    return sortAsc ? (av > bv ? 1 : -1) : (av < bv ? 1 : -1);
  }});

  tbody.innerHTML = data.map(r => `
    <tr>
      <td class="mono">${{r.resnum}}</td>
      <td class="mono">${{r.aa || ""}}</td>
      <td class="mono">${{r.rsa != null ? r.rsa.toFixed(3) : "—"}}</td>
      <td><span class="pill ${{r.ss}}">${{r.ss}}</span></td>
      <td class="mono">${{r.plddt != null ? r.plddt.toFixed(1) : "—"}}</td>
      <td><span class="pill ${{r.buried}}">${{r.buried || "—"}}</span></td>
      <td style="color:var(--soft);font-size:12px">${{r.domain}}</td>
    </tr>`).join("");

  countEl.textContent = `${{data.length.toLocaleString()}} residues`;
  emptyEl.style.display = data.length ? "none" : "block";
}}

document.querySelectorAll("th[data-k]").forEach(th =>
  th.addEventListener("click", () => {{
    if (sortKey === th.dataset.k) sortAsc = !sortAsc;
    else {{ sortKey = th.dataset.k; sortAsc = true; }}
    render();
  }})
);
["q","ss","dom"].forEach(id =>
  document.getElementById(id).addEventListener("input", render)
);
render();
</script></body></html>"""

print("Template defined.")

Template defined.


In [9]:
import json
from pathlib import Path

full_dir = Path("/content/rsa_full")
html_dir = Path("/content/html/biophysical")
html_dir.mkdir(parents=True, exist_ok=True)

for rsa_path in sorted(full_dir.glob("*.csv")):
    gene    = rsa_path.stem
    acc_row = acc_df[acc_df["gene"] == gene].iloc[0]

    accession    = acc_row["accession"]
    panel        = acc_row["panel"]
    length       = int(acc_row["length"])
    protein_name = acc_row["entry_name"]

    rsa_df       = pd.read_csv(rsa_path)
    residue_data = rsa_df[["resnum","aa","ss","rsa","plddt"]].to_dict(orient="records")

    gene_feats   = feat_df[feat_df["gene"] == gene]
    domain_data  = (gene_feats[["feature_type","description","start","end"]]
                    .rename(columns={"feature_type":"type"})
                    .dropna(subset=["start","end"])
                    .to_dict(orient="records"))
    domain_names = sorted(gene_feats[gene_feats["feature_type"]=="Domain"]["description"]
                          .dropna().unique())
    domain_options = "\n".join(f'<option value="{d}">{d}</option>' for d in domain_names)

    html = template.format(
        gene=gene, protein_name=protein_name, panel=panel,
        accession=accession, length=length,
        domain_options=domain_options,
        residue_json=json.dumps(residue_data),
        domain_json=json.dumps(domain_data),
    )

    (html_dir / f"{gene}.html").write_text(html, encoding="utf-8")
    print(f"{gene:<12} → {gene}.html")

FRAS1        → FRAS1.html
FREM2        → FREM2.html
TNXB         → TNXB.html
